<div style="padding:22px 26px;border:1px solid #dbe4ee;border-left:6px solid #176b87;border-radius:12px;background:linear-gradient(120deg,#f8fbfd,#edf7f6)">
<div style="font-size:12px;font-weight:700;letter-spacing:.08em;color:#d1495b">VERSION 5</div>
<h1 style="margin:3px 0 0;color:#102a43">RSNA Knee Baseline</h1>
<p style="margin:8px 0 0;color:#486581"><b>weak-label 2.5D student</b> — Use all 4,407 studies: reports supervise training, while inference remains image- and protocol-only.</p>
</div>


## Version map

| Version | Focus | Role |
|---|---|---|
| **V1** | audit | dataset structure and failure modes |
| **V2** | metadata | protocol-only sanity baseline |
| **V3** | teacher | report pseudo-labels without text leakage |
| **V4** | images | gold-only 2.5D model |
| **V5** ← current | student | weak-label 2.5D model + TTA |

In [ ]:
import os, re, gc, random, hashlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

class CFG:
    seed = 42
    targets = ['ACL','MCL','Medial Meniscus','Lateral Meniscus',
               'Medial OA','Lateral OA','PF OA','Effusion',
               'Synovitis',"Baker's",'Contusion','Fracture']
    work = Path('/kaggle/working') if Path('/kaggle').exists() else Path('work/notebook-output')

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

def find_root():
    candidates = [
        Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
        Path('/kaggle/input/rsna-knee-abnormality-detection'),
        Path('work/competition-csv'),
    ]
    candidates += list(Path('/kaggle/input').glob('*knee*')) if Path('/kaggle/input').exists() else []
    for p in candidates:
        if (p / 'train.csv').exists():
            return p
    raise FileNotFoundError('Competition data not found')

seed_everything(CFG.seed)
ROOT = find_root()
CFG.work.mkdir(parents=True, exist_ok=True)

PALETTE = ['#176b87','#64ccc5','#dafffb','#f5b971','#d1495b']
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titleweight': 'bold', 'axes.edgecolor': '#bcccdc',
    'grid.color': '#e6eef4', 'figure.facecolor': 'white',
    'axes.grid': True, 'axes.axisbelow': True
})
print('data:', ROOT)

In [ ]:
TARGETS = CFG.targets
train = pd.read_csv(ROOT / 'train.csv')
series = pd.read_csv(ROOT / 'train_series.csv')
test = pd.read_csv(ROOT / 'test.csv')
test_series = pd.read_csv(ROOT / 'test_series.csv')
sample = pd.read_csv(ROOT / 'sample_submission.csv')

has_gold = train[TARGETS].notna().all(axis=1)
gold = train.loc[has_gold].reset_index(drop=True)

facts = [
    ('train studies', f'{len(train):,}'),
    ('gold studies', f'{has_gold.sum():,}'),
    ('report-only', f'{(~has_gold).sum():,}'),
    ('series', f'{len(series):,}'),
    ('targets', len(TARGETS)),
    ('visible test', len(test)),
]
cards = ''.join(
    f'<div style="padding:12px 15px;border:1px solid #dbe4ee;border-radius:10px;min-width:105px">'
    f'<div style="font-size:20px;font-weight:700;color:#176b87">{v}</div>'
    f'<div style="font-size:11px;color:#627d98">{k}</div></div>' for k,v in facts
)
display(HTML(f'<div style="display:flex;gap:8px;flex-wrap:wrap">{cards}</div>'))

assert list(sample.columns) == ['StudyInstanceUID'] + TARGETS
assert series['StudyInstanceUID'].nunique() == len(train)
print('reports at test:', 'Report' in test.columns)
print('PatientSex in train.csv:', 'PatientSex' in train.columns)

## Data decisions

The final version keeps the audit close to the model: sparse gold labels, train-only reports, duplicated protocol flags and report fingerprints all change either supervision or validation.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))

pos = gold[TARGETS].sum().sort_values()
ax[0].barh(pos.index, pos.values, color=PALETTE[0])
ax[0].set(title='Gold positives', xlabel=f'count among {len(gold)}')
for i, v in enumerate(pos.values):
    ax[0].text(v + .35, i, str(int(v)), va='center', fontsize=8)

n_series = series.groupby('StudyInstanceUID').size()
bins = np.arange(n_series.min() - .5, n_series.max() + 1.5)
ax[1].hist(n_series, bins=bins, color=PALETTE[1], edgecolor='white')
ax[1].axvline(n_series.median(), color=PALETTE[4], ls='--', label=f'median {n_series.median():.0f}')
ax[1].set(title='Series per study', xlabel='series', ylabel='studies')
ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()

print(f'gold coverage: {has_gold.mean():.2%}')
print('gold studies with zero positives:', int((gold[TARGETS].sum(axis=1) == 0).sum()))
print('all studies contain planes:', series.groupby('StudyInstanceUID').Anatomical_Plane.nunique().value_counts().to_dict())
print('Fluid_Sensitive == Fat_Suppression:', bool(series.Fluid_Sensitive.equals(series.Fat_Suppression)))

In [ ]:
def normalize_report(text):
    text = re.sub(r'\s+', ' ', str(text).lower()).strip()
    return re.sub(r'\b\d+(?:[.,]\d+)?\b', '<n>', text)

report_key = train.Report.map(normalize_report)
groups = pd.DataFrame({'key': report_key, 'gold': has_gold}).groupby('key').gold.agg(['size','any','all'])
cross = groups[(groups['size'] > 1) & groups['any'] & ~groups['all']]

slot = series.assign(slot=series.Anatomical_Plane.str[0] + '_' + series.Fluid_Sensitive.astype(str))
slot = pd.crosstab(slot.StudyInstanceUID, slot.slot).reindex(
    columns=['A_0','A_1','C_0','C_1','S_0','S_1'], fill_value=0)
coverage = (slot > 0).mean().sort_values()

fig, ax = plt.subplots(figsize=(7.6, 3.2))
ax.barh(coverage.index, coverage.values, color=PALETTE[0])
ax.set(xlim=(0,1.03), xlabel='share of studies', title='Protocol slot coverage')
for i, v in enumerate(coverage.values):
    ax.text(v + .012, i, f'{v:.0%}', va='center', fontsize=8)
plt.tight_layout(); plt.show()

print('normalized duplicate-report studies:', int(groups.loc[groups['size'] > 1, 'size'].sum()))
print('gold-to-unlabeled report groups:', len(cross))
print('visible test is only the scoring placeholder:', len(test) == 3)

### Findings that change the pipeline

- **58 / 4,407 studies are labeled**, and every gold study has at least one positive finding. Gold prevalence is selection-biased.
- Reports cover the full training set but disappear at inference. Use them as a **teacher**, never as a test feature.
- Every study contains sagittal, coronal and axial series. Fixed plane slots are a strong first design.
- `Fluid_Sensitive` and `Fat_Suppression` are identical across 24,371 rows. Keep one.
- Normalized report duplicates connect labeled and unlabeled studies. Group report fingerprints before validation.
- `test.csv` contains three placeholder studies. Train/test shift estimates from this sample are inconclusive.


## Report teacher

Multilingual rules anchor common findings and negations; character and word TF-IDF add local phrasing. Teacher probabilities supervise the unlabeled image studies, while all 58 gold rows retain their original targets and higher weight.

In [ ]:
def protocol_features(df, study_ids=None):
    x = df.copy()
    x['slot'] = x.Anatomical_Plane.str[0] + '_' + x.Fluid_Sensitive.astype(str)
    f = pd.crosstab(x.StudyInstanceUID, x.slot)
    slots = ['A_0','A_1','C_0','C_1','S_0','S_1']
    f = f.reindex(columns=slots, fill_value=0)
    agg = x.groupby('StudyInstanceUID').agg(
        n_series=('SeriesInstanceUID','size'),
        n_planes=('Anatomical_Plane','nunique'),
        n_fluid=('Fluid_Sensitive','sum'))
    f = f.join(agg).astype(np.float32)
    if study_ids is not None:
        f = f.reindex(study_ids, fill_value=0)
    return f

train_meta = protocol_features(series, train.StudyInstanceUID)
test_meta = protocol_features(test_series, test.StudyInstanceUID)
FEATURES = train_meta.columns.tolist()
display(train_meta.describe().T.round(2))

In [ ]:
def save_submission(pred, name='submission.csv'):
    pred = np.asarray(pred, dtype=np.float64)
    assert pred.shape == (len(test), len(TARGETS))
    assert np.isfinite(pred).all()
    out = sample[['StudyInstanceUID']].copy()
    out[TARGETS] = np.clip(pred, 1e-5, 1 - 1e-5)
    path = CFG.work / name
    out.to_csv(path, index=False)
    assert list(out.columns) == ['StudyInstanceUID'] + TARGETS
    print(path, out.shape, f'[{out[TARGETS].min().min():.4f}, {out[TARGETS].max().max():.4f}]')
    return out


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy import sparse

TERMS = {
    'ACL': r'\bacl\b|anterior cruciate|ligament croise anterieur|\blca\b|vorder.{0,8}kreuzband',
    'MCL': r'\bmcl\b|medial collateral|ligament collateral medial|innenband',
    'Medial Meniscus': r'medial menisc|menisco medial|menisque interne|innenmeniskus',
    'Lateral Meniscus': r'lateral menisc|menisco lateral|menisque externe|aussenmeniskus|außenmeniskus',
    'Medial OA': r'medial.{0,35}(osteoarth|arthros|chondr|degenerat|joint space)',
    'Lateral OA': r'lateral.{0,35}(osteoarth|arthros|chondr|degenerat|joint space)',
    'PF OA': r'patellofemoral|femoropatel|retropatell|chondromalacia',
    'Effusion': r'effusion|joint fluid|epanchement|erguss|derrame articular',
    'Synovitis': r'synovit|sinovit|synovial thick|synovial hypertroph',
    "Baker's": r'baker.{0,8}cyst|popliteal cyst|kyste poplite|baker.?zyste',
    'Contusion': r'contusion|bone bruise|bone marrow (edema|oedema)|knochenmark',
    'Fracture': r'fractur|fraktur|fractura|avulsion|segond',
}
NEG = re.compile(r'\b(no|not|without|absence|negative|normal|intact|kein|keine|geen|sans|aucun|sin|yok)\b', re.I)

def rule_scores(text):
    text = re.sub(r'\s+', ' ', str(text).lower())
    out = []
    for target in TARGETS:
        hits = list(re.finditer(TERMS[target], text, re.I))
        if not hits:
            out.append(.08); continue
        asserted = False
        for h in hits:
            window = text[max(0, h.start()-55):min(len(text), h.end()+55)]
            if not NEG.search(window):
                asserted = True; break
        out.append(.88 if asserted else .12)
    return out

texts = train.Report.fillna('').str.lower().values
rules = np.asarray([rule_scores(t) for t in texts], dtype=np.float32)
char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=3, max_features=120_000, sublinear_tf=True)
word = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=70_000, sublinear_tf=True)
X_text = sparse.hstack([char.fit_transform(texts), word.fit_transform(texts), sparse.csr_matrix(rules)]).tocsr()
gold_idx = np.flatnonzero(has_gold.values)
teacher = np.zeros((len(train), len(TARGETS)), dtype=np.float32)
teacher_oof = np.zeros((len(gold), len(TARGETS)), dtype=np.float32)
teacher_rows = []

for j, target in enumerate(TARGETS):
    y = gold[target].astype(int).values
    cv = StratifiedKFold(4, shuffle=True, random_state=CFG.seed)
    for tr_idx, va_idx in cv.split(gold_idx, y):
        m = LogisticRegression(C=2, class_weight='balanced', solver='liblinear', max_iter=700)
        m.fit(X_text[gold_idx[tr_idx]], y[tr_idx])
        teacher_oof[va_idx, j] = m.predict_proba(X_text[gold_idx[va_idx]])[:,1]
    m = LogisticRegression(C=2, class_weight='balanced', solver='liblinear', max_iter=700)
    m.fit(X_text[gold_idx], y)
    teacher[:, j] = .75 * m.predict_proba(X_text)[:,1] + .25 * rules[:,j]
    teacher_rows.append((target, roc_auc_score(y, teacher_oof[:,j])))

teacher[gold_idx] = gold[TARGETS].values
teacher_score = pd.DataFrame(teacher_rows, columns=['target','text_oof_auc'])
print('teacher OOF macro:', round(teacher_score.text_oof_auc.mean(), 3))
display(teacher_score.round(3))

view = teacher_score.sort_values('text_oof_auc')
fig, ax = plt.subplots(figsize=(7.2,3.6))
ax.barh(view.target, view.text_oof_auc, color=[PALETTE[4] if v < .5 else PALETTE[0] for v in view.text_oof_auc])
ax.axvline(.5, color='#627d98', ls='--')
ax.set(xlim=(.3,1), xlabel='OOF AUC on 58 gold studies', title='Report teacher: useful, uneven, noisy')
plt.tight_layout(); plt.show()

pseudo = train[['StudyInstanceUID']].copy()
pseudo[TARGETS] = teacher
pseudo['is_hard'] = has_gold.astype(int)
pseudo.to_csv(CFG.work / 'pseudo_labels.csv', index=False)

## Image student

The student reads two 2.5D triplets from each anatomical plane. Gold studies are repeated, soft-label rows are down-weighted, and horizontal TTA stabilizes inference. A 12% metadata blend adds cheap model diversity without introducing train-only text.

In [ ]:
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from functools import lru_cache

class IMG:
    size = 192
    triplets = 3
    batch = 3
    workers = 2
    epochs = 8
    lr = 3e-4
    wd = 1e-3
    backbone = 'resnet18'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    amp = torch.cuda.is_available()

torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
print('device:', IMG.device)

In [ ]:
PLANES = ['Sagittal','Coronal','Axial']

@lru_cache(maxsize=18000)
def ordered_paths(folder_text):
    folder = Path(folder_text)
    rows = []
    for i, path in enumerate(folder.glob('*.dcm')):
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
            if hasattr(ds, 'ImageOrientationPatient') and hasattr(ds, 'ImagePositionPatient'):
                orient = np.asarray(ds.ImageOrientationPatient, dtype=float)
                normal = np.cross(orient[:3], orient[3:])
                key = float(np.dot(np.asarray(ds.ImagePositionPatient, dtype=float), normal))
            else:
                key = float(getattr(ds, 'InstanceNumber', i))
            rows.append((key, str(path)))
        except Exception:
            rows.append((float(i), str(path)))
    return tuple(p for _, p in sorted(rows))

def decode_triplets(folder, count=3, size=192):
    paths = ordered_paths(str(folder))
    if not paths:
        raise FileNotFoundError(folder)
    centers = np.linspace(0, len(paths)-1, count+2)[1:-1].round().astype(int)
    images = []
    for c in centers:
        ids = [max(0,c-1), c, min(len(paths)-1,c+1)]
        stack = []
        for i in ids:
            ds = pydicom.dcmread(paths[i], force=True)
            x = ds.pixel_array.astype(np.float32)
            x = x * float(getattr(ds, 'RescaleSlope', 1.0)) + float(getattr(ds, 'RescaleIntercept', 0.0))
            stack.append(x)
        x = np.stack(stack)
        lo, hi = np.percentile(x, [.5, 99.5])
        x = np.clip((x-lo)/(hi-lo+1e-6), 0, 1).astype(np.float32, copy=False)
        x = torch.from_numpy(x).unsqueeze(0)
        x = F.interpolate(x, (size,size), mode='bilinear', align_corners=False).squeeze(0)
        images.append(x)
    return torch.stack(images)

def choose_series(frame):
    if frame.empty:
        return [None] * len(PLANES)
    chosen = []
    for plane in PLANES:
        q = frame[frame.Anatomical_Plane.eq(plane)].sort_values(
            ['Fluid_Sensitive','SeriesInstanceUID'], ascending=[False,True])
        chosen.append(None if q.empty else q.iloc[0])
    return chosen

class KneeDataset(Dataset):
    def __init__(self, frame, series_frame, image_root, targets=None, sample_weight=None, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.groups = {k:v for k,v in series_frame.groupby('StudyInstanceUID')}
        self.image_root = Path(image_root)
        self.targets = targets
        self.sample_weight = sample_weight
        self.augment = augment

    def __len__(self): return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        uid = row.StudyInstanceUID
        items, plane_ids, masks = [], [], []
        for plane_id, item in enumerate(choose_series(self.groups.get(uid, pd.DataFrame()))):
            try:
                folder = self.image_root / uid / item.SeriesInstanceUID
                x = decode_triplets(folder, IMG.triplets, IMG.size)
                valid = True
            except Exception:
                x = torch.zeros(IMG.triplets,3,IMG.size,IMG.size)
                valid = False
            if self.augment and random.random() < .5:
                x = torch.flip(x, dims=[-1])
            items.append(x)
            plane_ids.extend([plane_id] * IMG.triplets)
            masks.extend([valid] * IMG.triplets)
        x = torch.cat(items, dim=0)
        y = torch.zeros(len(TARGETS)) if self.targets is None else torch.tensor(self.targets[idx], dtype=torch.float32)
        w = 1.0 if self.sample_weight is None else float(self.sample_weight[idx])
        return x, torch.tensor(plane_ids), torch.tensor(masks), y, torch.tensor(w), uid

class KneeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model(IMG.backbone, pretrained=False, num_classes=0, global_pool='avg')
        dim = self.encoder.num_features
        self.plane = nn.Embedding(3, dim)
        self.attn = nn.Sequential(nn.Linear(dim,128), nn.Tanh(), nn.Linear(128,1))
        self.head = nn.Sequential(nn.Dropout(.3), nn.Linear(dim,len(TARGETS)))

    def forward(self, x, plane, mask):
        b,k,c,h,w = x.shape
        f = self.encoder(x.reshape(b*k,c,h,w)).reshape(b,k,-1)
        f = f + self.plane(plane)
        a = self.attn(f).squeeze(-1).masked_fill(~mask, -1e4)
        z = (f * a.softmax(-1).unsqueeze(-1)).sum(1)
        return self.head(z)

print('DICOM loader: geometric ordering -> percentile scaling -> 2.5D triplets')

In [ ]:
def fit_image_model(train_frame, targets, weights, epochs=None):
    epochs = IMG.epochs if epochs is None else epochs
    ds = KneeDataset(train_frame, series, ROOT/'train_series', targets, weights, augment=True)
    loader = DataLoader(ds, batch_size=IMG.batch, shuffle=True, num_workers=IMG.workers,
                        pin_memory=True, persistent_workers=IMG.workers > 0)
    model = KneeNet().to(IMG.device)
    opt = torch.optim.AdamW(model.parameters(), lr=IMG.lr, weight_decay=IMG.wd)
    pos = np.clip(np.asarray(targets).sum(0), 1, None)
    neg = len(targets) - pos
    pos_weight = torch.tensor(np.clip(neg/pos,1,8), dtype=torch.float32, device=IMG.device)
    scaler = torch.cuda.amp.GradScaler(enabled=IMG.amp)

    for epoch in range(epochs):
        model.train(); losses=[]
        for x, plane, mask, y, w, _ in loader:
            x,plane,mask,y,w = x.to(IMG.device),plane.to(IMG.device),mask.to(IMG.device),y.to(IMG.device),w.to(IMG.device)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=IMG.amp):
                logits = model(x,plane,mask)
                raw = F.binary_cross_entropy_with_logits(logits,y,pos_weight=pos_weight,reduction='none').mean(1)
                loss = (raw*w).sum()/w.sum().clamp_min(1)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt); scaler.update(); losses.append(loss.item())
        print(f'epoch {epoch+1:02d}/{epochs} loss={np.mean(losses):.4f}')
    return model

@torch.no_grad()
def predict_image(model, flip=False):
    ds = KneeDataset(test, test_series, ROOT/'test_series')
    loader = DataLoader(ds, batch_size=IMG.batch, shuffle=False, num_workers=IMG.workers,
                        pin_memory=True, persistent_workers=IMG.workers > 0)
    model.eval(); pred=[]; ids=[]
    for x,plane,mask,_,_,uid in loader:
        x,plane,mask = x.to(IMG.device),plane.to(IMG.device),mask.to(IMG.device)
        with torch.cuda.amp.autocast(enabled=IMG.amp):
            p = torch.sigmoid(model(x,plane,mask))
            if flip:
                p = .5 * (p + torch.sigmoid(model(torch.flip(x,[-1]),plane,mask)))
        pred.append(p.float().cpu().numpy()); ids.extend(uid)
    return np.concatenate(pred), ids

In [ ]:
# hard rows are repeated so 58 audited targets remain visible to the optimizer
target_frame = train[['StudyInstanceUID']].copy()
soft_y = teacher.astype(np.float32)
soft_w = np.full(len(train), .35, dtype=np.float32)
soft_w[has_gold.values] = 1.0

hard_repeat = 8
hard_frame = pd.concat([train.loc[has_gold, ['StudyInstanceUID']]] * hard_repeat, ignore_index=True)
hard_targets = np.tile(gold[TARGETS].values.astype(np.float32), (hard_repeat,1))

mix_frame = pd.concat([target_frame, hard_frame], ignore_index=True)
mix_targets = np.concatenate([soft_y, hard_targets])
mix_weights = np.concatenate([soft_w, np.ones(len(hard_frame), dtype=np.float32)])

IMG.epochs = 2
IMG.triplets = 2
model = fit_image_model(mix_frame, mix_targets, mix_weights, epochs=IMG.epochs)
image_pred, ids = predict_image(model, flip=True)
assert ids == test.StudyInstanceUID.tolist()

# low-weight metadata blend: cheap diversity, not calibration
X_all = train_meta.values; Xt = test_meta.values
meta_pred = np.zeros_like(image_pred)
for j in range(len(TARGETS)):
    m = make_pipeline(StandardScaler(), Ridge(alpha=8.0))
    m.fit(X_all, teacher[:,j], ridge__sample_weight=np.where(has_gold.values,8.0,1.0))
    meta_pred[:,j] = m.predict(Xt)

final_pred = .88*image_pred + .12*np.clip(meta_pred,.02,.98)
torch.save(model.state_dict(), CFG.work/'knee_v5.pt')
submission = save_submission(final_pred)
submission.head()